In [1]:
#Importing Libraries

from collections import Counter
import json
from pathlib import Path
import random
import pandas as pd

In [2]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
PROJECT_ROOT = BASE_DIR / "Data" / "Processed"
SPLIT_DATA = BASE_DIR / "Data" / "Splits"
print(BASE_DIR)
print(RAW_DIR)
print(PROJECT_ROOT)
print(SPLIT_DATA)

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Splits


In [3]:
#Access CSV FILE and SPlit data into 3 parts, 80/10/10

df = pd.read_csv(PROJECT_ROOT / "bio_dataset.csv")

# unique record IDs ** split by record_id to avoid leakage across splits. Sorting ensures deterministic behavior before shuffling.
record_ids = sorted(df["record_id"].unique()) #split by record, not by rows. collect all unique record IDs.Sorting ensures deterministic ordering before shuffling.

random.seed(42) #Sets a fixed seed so the shuffle always produces the same order.
random.shuffle(record_ids) #Shuffling removes ordering bias. If the dataset is sorted by topic or time, splitting without shuffling would create biased splits.

# compute index boundaries based on percentages so the split scales automatically with dataset size
n = len(record_ids)
train_end = int(n * 0.8)
val_end = train_end + int(n * 0.1)

# Slice the shuffled IDs into sets
# slice the shuffled list into 80/10/10 segments. Using sets makes filtering efficient and avoids accidental duplicates
train_ids = set(record_ids[:train_end])
val_ids   = set(record_ids[train_end:val_end])
test_ids  = set(record_ids[val_end:])

# Filter the dataframe by record_id
# filter the BIO dataframe using the ID sets. This guarantees that all tokens from a record stay in the same split, preventing leakage.
train_df = df[df["record_id"].isin(train_ids)]
val_df   = df[df["record_id"].isin(val_ids)]
test_df  = df[df["record_id"].isin(test_ids)]

## Verify

In [4]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(62076, 3)
(7703, 3)
(7748, 3)


In [5]:
print(len(train_ids))
print(len(val_ids))
print(len(test_ids))

4000
500
500


In [6]:
print(
    len(train_ids & val_ids),
    len(train_ids & test_ids),
    len(val_ids & test_ids)
)

0 0 0


In [7]:
#Save train_ids, val_ids, and test_ids as csv

import pandas as pd

df = pd.DataFrame(train_ids)
df.to_csv(SPLIT_DATA/'train_ids.csv', index=False)

import pandas as pd

df = pd.DataFrame(val_ids)
df.to_csv(SPLIT_DATA/'val_ids.csv', index=False)

df = pd.DataFrame(test_ids)
df.to_csv(SPLIT_DATA/'test_ids.csv', index=False)

In [8]:
# Verify

print(len(train_ids))
print(len(val_ids))
print(len(test_ids))

4000
500
500


In [9]:
df = pd.read_csv(SPLIT_DATA/'train_ids.csv')
df[:5]

,0
0,1
1,2
2,3
3,6
4,7


In [10]:
# Load json file

json_path = RAW_DIR / "synthetic_notes.json"

with open(json_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [11]:
# create the split records:

import pandas as pd

# 1. Read Train IDs
train_ids = set(pd.read_csv(SPLIT_DATA / 'train_ids.csv')["0"])
train_records = [r for r in data if r["id"] in train_ids]

# 2. Read Test IDs (Fixed pd.read_csv and closing parenthesis)
test_ids = set(pd.read_csv(SPLIT_DATA / 'test_ids.csv')["0"])
test_records  = [r for r in data if r["id"] in test_ids]

# 3. Read Val IDs (Fixed pd.read_csv and closing parenthesis)
val_ids = set(pd.read_csv(SPLIT_DATA / 'val_ids.csv')["0"])
val_records   = [r for r in data if r["id"] in val_ids]


In [12]:
# Verify

print(train_records[:1])
print('\n')
print(test_records[:1])
print('\n')
print(val_records[:1])

[{'id': 1, 'text': 'Known case of Influenza. Advised to take Budesonide 250 mg once daily.', 'entities': [{'start': 14, 'end': 23, 'label': 'DISEASE'}, {'start': 41, 'end': 51, 'label': 'MEDICATION'}, {'start': 52, 'end': 69, 'label': 'DOSAGE'}]}]


[{'id': 4, 'text': 'Patient returns for review of alzheimer disease. Reports improvement in blood in urine. Maintain Omeprazole 10 ml twice daily.', 'entities': [{'start': 72, 'end': 86, 'label': 'SYMPTOM'}, {'start': 30, 'end': 47, 'label': 'DISEASE'}, {'start': 97, 'end': 107, 'label': 'MEDICATION'}, {'start': 108, 'end': 125, 'label': 'DOSAGE'}]}]


[{'id': 10, 'text': 'Patient suffers from Dvt. Advised to take Amoxicillin 250 mg once daily.', 'entities': [{'start': 21, 'end': 24, 'label': 'DISEASE'}, {'start': 42, 'end': 53, 'label': 'MEDICATION'}, {'start': 54, 'end': 71, 'label': 'DOSAGE'}]}]


In [13]:
# Verify

df = pd.read_csv(PROJECT_ROOT / "bio_dataset.csv")

print(type(data))
print(type(df))

print(len(data))
print(df.shape)

<class 'list'>
<class 'pandas.core.frame.DataFrame'>
5000
(77527, 3)


### Save split records

In [14]:
import json

# Save train
with open(SPLIT_DATA / "train_records.json", "w", encoding="utf-8") as f:
    json.dump(train_records, f, indent=4)

# Save val
with open(SPLIT_DATA / "val_records.json", "w", encoding="utf-8") as f:
    json.dump(val_records, f, indent=4)

# Save test
with open(SPLIT_DATA / "test_records.json", "w", encoding="utf-8") as f:
    json.dump(test_records, f, indent=4)
